# RSNA Knee Abnormality Detection — Semi-Supervised Adaptive CLIP Mean Teacher

This single-run Kaggle submission preserves the paper's CLIP ViT-B/16 architecture—residual adapters at blocks 3/7/11, linear multi-depth fusion, learnable training-only text anchors, and a linear classifier—while adding a clearly separated **Mean Teacher semi-supervised extension**.

Training uses **46 labelled gold MRI studies + 1,000 genuinely unlabelled MRI studies**. The unlabelled studies are never assigned fixed report labels. An EMA teacher instead supplies online MRI consistency targets. The final 12 gold studies remain untouched until the one-time final evaluation.


In [1]:
# Configuration and imports — no pip install and no network access
from pathlib import Path
import copy, gc, hashlib, inspect, json, math, os, random, time, warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, CLIPModel
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import cv2
import pydicom
from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 2026
N_GOLD_DEV = 46
N_FINAL_HOLDOUT = 12
N_UNLABELED_STUDIES = 1000
SERIES_PER_STUDY = 3

IMAGE_SIZE = 224
MAX_SLICES = 10
SLICE_CHUNK = 4
NUM_WORKERS = 2

# Paper architecture.
ADAPTER_LAYER_INDICES = (2, 6, 10)  # zero-based -> paper blocks 3, 7, 11
ADAPTER_BOTTLENECK_RATIO = 0.25
ADAPTER_GAMMA_INIT = 1e-3
PROMPT_CONTEXT_TOKENS = 4
ALIGNMENT_DIM = 512
ALIGNMENT_TEMPERATURE = 0.07
ALIGNMENT_BETA = 0.10
PE_L2_COEFF = 1e-6

# Semi-supervised extension.
WARMUP_EPOCHS = 2
SSL_EPOCHS = 1
IMAGE_LR = 1e-4
IMAGE_WEIGHT_DECAY = 1e-5
EMA_DECAY = 0.99
UNLABELED_WEIGHT_MAX = 1.0
POSITIVE_CONFIDENCE = 0.70
NEGATIVE_CONFIDENCE = 0.30

LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

OUT = Path("/kaggle/working/rsna_adaptive_clip_mean_teacher")
OUT.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)
if AMP:
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


## 1. Validate exact offline input paths

Exact paths avoid slow recursive scans and eliminate ambiguity when other Kaggle inputs contain similarly named files. Hugging Face may store weights as `pytorch_model.bin`, sharded `.bin` files, or `model.safetensors`; `from_pretrained(..., local_files_only=True)` handles all supported formats.


In [2]:
# Exact Kaggle mount points. Change only CLIP_MODEL_DIR if Kaggle displays a different slug/version.
ROOT_CANDIDATES = [
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
]
ROOT = next((path for path in ROOT_CANDIDATES if path.exists()), ROOT_CANDIDATES[0])
CLIP_MODEL_DIR = Path("/kaggle/input/models/amritgopinath/clip-vit-model/pytorch/default/1")
TRAIN_SERIES_ROOT = ROOT / "train_series"

required_paths = {
    "competition root": ROOT,
    "train.csv": ROOT / "train.csv",
    "train_series.csv": ROOT / "train_series.csv",
    "sample_submission.csv": ROOT / "sample_submission.csv",
    "train_series directory": TRAIN_SERIES_ROOT,
    "CLIP ViT-B/16 directory": CLIP_MODEL_DIR,
    "CLIP config": CLIP_MODEL_DIR / "config.json",
}
missing = {name: str(path) for name, path in required_paths.items() if not path.exists()}
if missing:
    raise FileNotFoundError(
        "Required Kaggle inputs are missing. Attach the competition data and full "
        f"openai/clip-vit-base-patch16 model. Missing: {missing}"
    )

clip_weights = (
    list(CLIP_MODEL_DIR.glob("*.safetensors"))
    + list(CLIP_MODEL_DIR.glob("pytorch_model*.bin"))
    + list(CLIP_MODEL_DIR.glob("*.index.json"))
)
if not clip_weights:
    raise FileNotFoundError(f"No CLIP weights found in {CLIP_MODEL_DIR}")

train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
train_series = pd.read_csv(
    ROOT / "train_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
).drop_duplicates().copy()
sample_submission = pd.read_csv(ROOT / "sample_submission.csv", dtype={"StudyInstanceUID": str})

expected_columns = ["StudyInstanceUID"] + LABEL_COLS
if list(sample_submission.columns) != expected_columns:
    raise ValueError(
        "Unexpected submission schema. "
        f"Expected {expected_columns}, received {sample_submission.columns.tolist()}"
    )
if not train["StudyInstanceUID"].is_unique:
    raise ValueError("train.csv must contain one row per study")

print("Competition root:", ROOT)
print("CLIP directory:", CLIP_MODEL_DIR)
print("CLIP weights:", [path.name for path in clip_weights])
print("train:", train.shape, "train_series:", train_series.shape, "submission template:", sample_submission.shape)
display(sample_submission.head())


Competition root: /kaggle/input/competitions/rsna-knee-abnormality-detection
CLIP directory: /kaggle/input/models/amritgopinath/clip-vit-model/pytorch/default/1
CLIP weights: ['pytorch_model.bin']
train: (4407, 14) train_series: (24371, 5) submission template: (3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


## 2. Fixed, balanced 46/12 gold split

The split is deterministic from `SEED`. A bounded candidate search chooses 12 holdout studies whose label prevalence is close to the full gold set while giving absolute priority to preserving both classes in the 46-study teacher-development set. The resulting assignment is exported and never changed downstream.

In [3]:
gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"] + LABEL_COLS].copy().reset_index(drop=True)
non_gold = train.loc[~gold_mask].copy().reset_index(drop=True)

if len(gold) != N_GOLD_DEV + N_FINAL_HOLDOUT:
    raise RuntimeError(
        f"This locked protocol expects {N_GOLD_DEV + N_FINAL_HOLDOUT} fully labelled studies, found {len(gold)}. "
        "Review the competition data before changing the protocol."
    )

Y_gold = gold[LABEL_COLS].astype(int).to_numpy()

def choose_holdout(y, n_holdout, seed, trials=50000):
    rng = np.random.default_rng(seed)
    n = len(y)
    full_prev = y.mean(axis=0)
    best = None
    for _ in range(trials):
        h = np.sort(rng.choice(n, n_holdout, replace=False))
        keep = np.ones(n, dtype=bool); keep[h] = False
        yh, yd = y[h], y[keep]
        # Teacher-dev must retain both classes wherever the full gold set has both.
        missing_dev = sum(len(np.unique(yd[:, j])) < len(np.unique(y[:, j])) for j in range(y.shape[1]))
        # Holdout class coverage is desirable but may be impossible for very rare labels.
        missing_holdout = sum(len(np.unique(yh[:, j])) < 2 for j in range(y.shape[1]))
        prevalence_error = np.abs(yh.mean(axis=0) - full_prev).mean()
        score = 1000 * missing_dev + 0.25 * missing_holdout + prevalence_error
        if best is None or score < best[0]:
            best = (score, h, missing_dev, missing_holdout, prevalence_error)
    return best

split_score, hold_idx, miss_dev, miss_holdout, prev_error = choose_holdout(
    Y_gold, N_FINAL_HOLDOUT, SEED
)
hold_mask = np.zeros(len(gold), dtype=bool); hold_mask[hold_idx] = True
gold_dev = gold.loc[~hold_mask].copy().reset_index(drop=True)
gold_final = gold.loc[hold_mask].copy().reset_index(drop=True)

assert len(gold_dev) == N_GOLD_DEV and len(gold_final) == N_FINAL_HOLDOUT
assert set(gold_dev.StudyInstanceUID).isdisjoint(set(gold_final.StudyInstanceUID))

split_table = pd.concat([
    gold_dev[["StudyInstanceUID"]].assign(split="teacher_dev"),
    gold_final[["StudyInstanceUID"]].assign(split="final_gold_holdout"),
], ignore_index=True)
split_table.to_csv(OUT / "gold_split_46_12.csv", index=False)

prevalence = pd.DataFrame({
    "pathology": LABEL_COLS,
    "all_58": gold[LABEL_COLS].mean().values,
    "teacher_dev_46": gold_dev[LABEL_COLS].mean().values,
    "final_holdout_12": gold_final[LABEL_COLS].mean().values,
    "holdout_positive_count": gold_final[LABEL_COLS].sum().astype(int).values,
})
print("Gold:", len(gold), "teacher-dev:", len(gold_dev), "final holdout:", len(gold_final))
print("Candidate-search diagnostics:", {"missing_dev_classes": miss_dev, "missing_holdout_classes": miss_holdout, "prevalence_MAE": prev_error})
display(prevalence)

Gold: 58 teacher-dev: 46 final holdout: 12
Candidate-search diagnostics: {'missing_dev_classes': 0, 'missing_holdout_classes': 0, 'prevalence_MAE': np.float64(0.024664750957854406)}


,pathology,all_58,teacher_dev_46,final_holdout_12,holdout_positive_count
0,ACL,0.413793,0.413043,0.416667,5
1,MCL,0.155172,0.152174,0.166667,2
2,Medial Meniscus,0.448276,0.434783,0.500000,6
3,Lateral Meniscus,0.396552,0.391304,0.416667,5
4,Medial OA,0.258621,0.260870,0.250000,3
5,Lateral OA,0.189655,0.195652,0.166667,2
6,PF OA,0.362069,0.347826,0.416667,5
7,Effusion,0.603448,0.608696,0.583333,7
8,Synovitis,0.465517,0.456522,0.500000,6
9,Baker's,0.206897,0.217391,0.166667,2


## 3. Select three series from 1,000 genuinely unlabelled studies

No reports or fixed pseudo-labels are used. Selection is label-independent: require three readable series, prefer one sagittal/coronal/axial series, and use acquisition-frequency ranking only as a deterministic fallback.


In [4]:
import os

def canonical_text(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper()

def boolish_text(x):
    if pd.isna(x):
        return "UNKNOWN"
    t = str(x).strip().lower()
    if t in {"true", "t", "yes", "y", "1", "1.0"}:
        return "TRUE"
    if t in {"false", "f", "no", "n", "0", "0.0"}:
        return "FALSE"
    return t.upper()

series_meta = train_series.copy()
for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
    if col not in series_meta.columns:
        series_meta[col] = "UNKNOWN"

series_meta["plane_norm"] = (
    series_meta["Anatomical_Plane"]
    .map(canonical_text)
    .replace({"SAG": "SAGITTAL", "COR": "CORONAL", "AX": "AXIAL"})
)
series_meta["fluid_norm"] = series_meta["Fluid_Sensitive"].map(boolish_text)
series_meta["fat_norm"] = series_meta["Fat_Suppression"].map(boolish_text)
series_meta["signature"] = (
    series_meta[["plane_norm", "fluid_norm", "fat_norm"]]
    .agg("|".join, axis=1)
)

non_gold_ids = set(non_gold["StudyInstanceUID"].astype(str))
eligible = series_meta[
    series_meta["StudyInstanceUID"].astype(str).isin(non_gold_ids)
].drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"]).copy()

if eligible.empty:
    raise RuntimeError("No non-gold series found in train_series.csv")

# Rank acquisition types using metadata. No DICOM directories are scanned here.
signature_counts = (
    eligible.groupby("signature")["StudyInstanceUID"]
    .nunique()
    .sort_values(ascending=False)
)
TARGET_SIGNATURE = signature_counts.index[0]
signature_rank = {
    signature: rank for rank, signature in enumerate(signature_counts.index)
}
eligible["signature_rank"] = eligible["signature"].map(signature_rank)
eligible["target_first"] = (
    eligible["signature"] != TARGET_SIGNATURE
).astype(int)

def series_dir_ok(row, root=TRAIN_SERIES_ROOT):
    directory = root / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID)
    try:
        with os.scandir(directory) as entries:
            return any(
                entry.is_file() and not entry.name.startswith(".")
                for entry in entries
            )
    except (FileNotFoundError, NotADirectoryError, PermissionError):
        return False

# Keep this function: the later gold/test cells also call it.
def choose_plane_diverse_series(frame):
    selected_groups = []
    for _, group in frame.groupby("StudyInstanceUID", sort=True):
        group = group.sort_values(
            ["target_first", "signature_rank", "SeriesInstanceUID"]
        )
        chosen_indices = []
        for plane in ["SAGITTAL", "CORONAL", "AXIAL"]:
            candidates = group[group["plane_norm"] == plane]
            if not candidates.empty:
                chosen_indices.append(candidates.index[0])
        for index in group.index:
            if len(chosen_indices) >= SERIES_PER_STUDY:
                break
            if index not in chosen_indices:
                chosen_indices.append(index)
        selected_groups.append(group.loc[chosen_indices[:SERIES_PER_STUDY]])

    return (
        pd.concat(selected_groups, ignore_index=True)
        if selected_groups else frame.iloc[0:0].copy()
    )

# Random order BEFORE filesystem checks: stop once 1,000 usable studies exist.
groups = {
    str(study_id): group.sort_values(
        ["target_first", "signature_rank", "SeriesInstanceUID"]
    )
    for study_id, group in eligible.groupby("StudyInstanceUID", sort=False)
    if len(group) >= SERIES_PER_STUDY
}

rng = np.random.default_rng(SEED + 11)
candidate_ids = rng.permutation(sorted(groups))

selected_ids = []
selected_rows = []

for examined, study_id in enumerate(candidate_ids, start=1):
    group = groups[study_id]
    chosen = []
    checked = {}

    def usable(index):
        if index not in checked:
            checked[index] = series_dir_ok(group.loc[index])
        return checked[index]

    # First try to get one readable series from each plane.
    for plane in ["SAGITTAL", "CORONAL", "AXIAL"]:
        candidates = group[group["plane_norm"] == plane]
        for index in candidates.index:
            if usable(index):
                chosen.append(index)
                break

    # Fill missing slots with other readable series.
    if len(chosen) < SERIES_PER_STUDY:
        for index in group.index:
            if index not in chosen and usable(index):
                chosen.append(index)
            if len(chosen) == SERIES_PER_STUDY:
                break

    if len(chosen) >= SERIES_PER_STUDY:
        selected_ids.append(study_id)
        selected_rows.append(group.loc[chosen[:SERIES_PER_STUDY]])

    if examined % 200 == 0:
        print(
            f"Checked {examined}/{len(candidate_ids)} candidate studies; "
            f"selected {len(selected_ids)}/{N_UNLABELED_STUDIES}",
            flush=True,
        )

    if len(selected_ids) == N_UNLABELED_STUDIES:
        break

if not selected_ids:
    raise RuntimeError("No non-gold studies have three readable MRI series")

if len(selected_ids) < N_UNLABELED_STUDIES:
    warnings.warn(
        f"Requested {N_UNLABELED_STUDIES} unlabelled studies, "
        f"but found only {len(selected_ids)} with three readable series."
    )

selected_unlabelled_ids = np.array(sorted(selected_ids))
selected_unlabelled_studies = pd.DataFrame({
    "StudyInstanceUID": selected_unlabelled_ids
})

unlabelled_series_selection = (
    pd.concat(selected_rows, ignore_index=True)
    .sort_values(
        ["StudyInstanceUID", "target_first",
         "signature_rank", "SeriesInstanceUID"]
    )
    .reset_index(drop=True)
)

assert (
    unlabelled_series_selection.groupby("StudyInstanceUID")
    .size()
    .eq(SERIES_PER_STUDY)
    .all()
)
assert set(selected_unlabelled_ids).isdisjoint(
    set(gold["StudyInstanceUID"].astype(str))
)

selection_cols = [
    "StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane",
    "Fluid_Sensitive", "Fat_Suppression", "signature"
]
unlabelled_series_selection[selection_cols].to_csv(
    OUT / "unlabelled_1000studies_3series_selection.csv",
    index=False,
)

print("Target signature:", TARGET_SIGNATURE)
print(
    f"Selected unlabelled studies: "
    f"{len(selected_unlabelled_studies)}/{N_UNLABELED_STUDIES} requested"
)
print("Selected unlabelled series:", len(unlabelled_series_selection))
display(signature_counts.head(10).rename("metadata_studies").to_frame())
display(unlabelled_series_selection[selection_cols].head(9))

Checked 200/4349 candidate studies; selected 200/1000
Checked 400/4349 candidate studies; selected 400/1000
Checked 600/4349 candidate studies; selected 600/1000
Checked 800/4349 candidate studies; selected 800/1000
Checked 1000/4349 candidate studies; selected 1000/1000
Target signature: AXIAL|TRUE|TRUE
Selected unlabelled studies: 1000/1000 requested
Selected unlabelled series: 3000


,metadata_studies
signature,
AXIAL|TRUE|TRUE,4349
SAGITTAL|FALSE|FALSE,4209
CORONAL|TRUE|TRUE,4192
SAGITTAL|TRUE|TRUE,4094
CORONAL|FALSE|FALSE,3361
AXIAL|FALSE|FALSE,844


,StudyInstanceUID,SeriesInstanceUID,Anatomical_Plane,Fluid_Sensitive,Fat_Suppression,signature
0,1.2.826.0.1.3680043.8.498.10085187975213640798...,1.2.826.0.1.3680043.8.498.62528033445367515360...,Axial,1,1,AXIAL|TRUE|TRUE
1,1.2.826.0.1.3680043.8.498.10085187975213640798...,1.2.826.0.1.3680043.8.498.56139651590917417326...,Sagittal,0,0,SAGITTAL|FALSE|FALSE
2,1.2.826.0.1.3680043.8.498.10085187975213640798...,1.2.826.0.1.3680043.8.498.11937405551646049903...,Coronal,0,0,CORONAL|FALSE|FALSE
3,1.2.826.0.1.3680043.8.498.10096216489317602387...,1.2.826.0.1.3680043.8.498.43172859976153629688...,Axial,1,1,AXIAL|TRUE|TRUE
4,1.2.826.0.1.3680043.8.498.10096216489317602387...,1.2.826.0.1.3680043.8.498.46174887318465327763...,Sagittal,0,0,SAGITTAL|FALSE|FALSE
5,1.2.826.0.1.3680043.8.498.10096216489317602387...,1.2.826.0.1.3680043.8.498.81693348047871174074...,Coronal,1,1,CORONAL|TRUE|TRUE
6,1.2.826.0.1.3680043.8.498.10118623027360228027...,1.2.826.0.1.3680043.8.498.25823386585191095678...,Axial,1,1,AXIAL|TRUE|TRUE
7,1.2.826.0.1.3680043.8.498.10118623027360228027...,1.2.826.0.1.3680043.8.498.17355150797297345667...,Sagittal,0,0,SAGITTAL|FALSE|FALSE
8,1.2.826.0.1.3680043.8.498.10118623027360228027...,1.2.826.0.1.3680043.8.498.12098721286482739004...,Coronal,1,1,CORONAL|TRUE|TRUE


## 4. Evaluation metrics

Metrics are defined before visual training. The 12-study final holdout is not used for early stopping, threshold selection, confidence tuning, or model selection.


In [5]:
def multilabel_metrics(y, p):
    rows = []
    for j, label in enumerate(LABEL_COLS):
        yt, pt = y[:, j].astype(int), p[:, j]
        two_classes = len(np.unique(yt)) == 2
        rows.append({
            "pathology": label,
            "n": len(yt),
            "positives": int(yt.sum()),
            "roc_auc": roc_auc_score(yt, pt) if two_classes else np.nan,
            "pr_auc": average_precision_score(yt, pt) if yt.sum() > 0 else np.nan,
            "f1_at_0.5": f1_score(yt, pt >= 0.5, zero_division=0),
        })
    metrics = pd.DataFrame(rows)
    macro = {
        "pathology": "MACRO",
        "n": int(metrics.n.sum()),
        "positives": int(metrics.positives.sum()),
        "roc_auc": metrics.roc_auc.mean(skipna=True),
        "pr_auc": metrics.pr_auc.mean(skipna=True),
        "f1_at_0.5": metrics["f1_at_0.5"].mean(skipna=True),
    }
    return pd.concat([metrics, pd.DataFrame([macro])], ignore_index=True)


## 5. Paper architecture: Adaptive CLIP-MIL

The CLIP backbone and text encoder remain frozen. Trainable components are the residual adapters, linear multi-depth fusion, class-specific prompt context, visual/text alignment projections, and 12-label classifier. The subsequent Mean Teacher stage is the semi-supervised extension.


In [6]:
def visible_files(folder):
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))

def geometric_scalar(dcm):
    ori = getattr(dcm, "ImageOrientationPatient", None)
    pos = getattr(dcm, "ImagePositionPatient", None)
    if ori is None or pos is None:
        return None
    try:
        ori = np.asarray(ori, dtype=float)
        pos = np.asarray(pos, dtype=float)
        normal = np.cross(ori[:3], ori[3:])
        norm = np.linalg.norm(normal)
        return None if norm == 0 else float(np.dot(pos, normal / norm))
    except Exception:
        return None

def ordered_dicom_paths(series_dir):
    records = []
    for path in visible_files(series_dir):
        try:
            dcm = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        except Exception:
            continue
        geom = geometric_scalar(dcm)
        instance = getattr(dcm, "InstanceNumber", None)
        try:
            instance = float(instance) if instance is not None else None
        except Exception:
            instance = None
        records.append((path, geom, instance))
    if records and all(r[1] is not None for r in records):
        records.sort(key=lambda r: r[1])
    elif records and all(r[2] is not None for r in records):
        records.sort(key=lambda r: r[2])
    else:
        records.sort(key=lambda r: r[0].name)
    return [r[0] for r in records]

def resize_pad(image, size=IMAGE_SIZE):
    h, w = image.shape
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size), dtype=np.float32)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas

def decode_dicom(path):
    dcm = pydicom.dcmread(str(path), force=True)
    x = dcm.pixel_array.astype(np.float32)
    if x.ndim > 2:
        x = np.squeeze(x)
    if x.ndim != 2:
        raise ValueError(f"Unsupported pixel shape {x.shape}")
    x = x * float(getattr(dcm, "RescaleSlope", 1.0)) + float(getattr(dcm, "RescaleIntercept", 0.0))
    if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
        x = x.max() - x
    return x

def load_series_volume(series_root, study_id, series_id, max_slices=MAX_SLICES, stochastic=False):
    paths = ordered_dicom_paths(series_root / str(study_id) / str(series_id))
    if not paths:
        raise ValueError(f"No readable DICOM headers for {study_id}/{series_id}")

    if stochastic:
        targets = []
        for bin_index in range(max_slices):
            low = min(len(paths) - 1, int(np.floor(bin_index * len(paths) / max_slices)))
            high = min(len(paths) - 1, int(np.floor((bin_index + 1) * len(paths) / max_slices)) - 1)
            high = max(low, high)
            targets.append(np.random.randint(low, high + 1))
        targets = np.asarray(targets, dtype=int)
    else:
        targets = np.linspace(0, len(paths) - 1, max_slices).round().astype(int)
    cache, raw = {}, []
    for target in targets:
        chosen = None
        offsets = [0]
        for distance in range(1, len(paths)):
            offsets.extend([-distance, distance])
        for offset in offsets:
            idx = int(target + offset)
            if idx < 0 or idx >= len(paths):
                continue
            if idx not in cache:
                try:
                    cache[idx] = decode_dicom(paths[idx])
                except Exception:
                    cache[idx] = None
            if cache[idx] is not None:
                chosen = cache[idx]
                break
        if chosen is not None:
            raw.append(chosen)

    if not raw:
        raise ValueError(f"No decodable pixel data for {study_id}/{series_id}")
    if len(raw) != max_slices:
        indices = np.linspace(0, len(raw) - 1, max_slices).round().astype(int)
        raw = [raw[i] for i in indices]

    finite_chunks = [x[np.isfinite(x)] for x in raw if np.isfinite(x).any()]
    if not finite_chunks:
        raise ValueError(f"No finite pixels for {study_id}/{series_id}")
    lo, hi = np.percentile(np.concatenate(finite_chunks), [1, 99])
    processed = []
    for x in raw:
        x = np.nan_to_num(x, nan=lo, posinf=hi, neginf=lo)
        x = (np.clip(x, lo, hi) - lo) / (hi - lo) if hi > lo else np.zeros_like(x)
        processed.append(resize_pad(x))
    return torch.from_numpy(np.stack(processed)[:, None].astype(np.float32))


CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)

ANCHOR_TEXT = [
    ("an intact anterior cruciate ligament", "an anterior cruciate ligament tear"),
    ("an intact medial collateral ligament", "a medial collateral ligament injury"),
    ("an intact medial meniscus", "a medial meniscus tear"),
    ("an intact lateral meniscus", "a lateral meniscus tear"),
    ("no medial compartment osteoarthritis", "medial compartment osteoarthritis"),
    ("no lateral compartment osteoarthritis", "lateral compartment osteoarthritis"),
    ("no patellofemoral osteoarthritis", "patellofemoral osteoarthritis"),
    ("no joint effusion", "joint effusion"),
    ("no synovitis", "synovitis"),
    ("no Baker cyst", "a Baker cyst"),
    ("no bone contusion", "a bone contusion"),
    ("no fracture", "a fracture"),
]


class ResidualAdapter(nn.Module):
    def __init__(self, width, ratio=ADAPTER_BOTTLENECK_RATIO):
        super().__init__()
        bottleneck = max(1, int(round(width * ratio)))
        self.norm = nn.LayerNorm(width, elementwise_affine=False)
        self.down = nn.Linear(width, bottleneck)
        self.up = nn.Linear(bottleneck, width)
        self.gamma = nn.Parameter(torch.tensor(float(ADAPTER_GAMMA_INIT)))
        nn.init.xavier_uniform_(self.down.weight)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden):
        return hidden + self.gamma * self.up(F.gelu(self.down(self.norm(hidden))))


class MultiDepthCLIPVisual(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.vision = clip_model.vision_model
        self.width = int(clip_model.config.vision_config.hidden_size)
        self.selected = tuple(ADAPTER_LAYER_INDICES)
        self.adapters = nn.ModuleDict({
            str(layer): ResidualAdapter(self.width)
            for layer in self.selected
        })
        self.fusion = nn.Linear(len(self.selected) * self.width, self.width)

    def _run_block(self, layer, hidden):
        parameters = inspect.signature(layer.forward).parameters
        if "causal_attention_mask" in parameters:
            output = layer(
                hidden,
                attention_mask=None,
                causal_attention_mask=None,
                output_attentions=False,
            )
        else:
            output = layer(hidden, attention_mask=None)
        return output[0] if isinstance(output, (tuple, list)) else output

    def forward(self, pixel_values):
        hidden = self.vision.embeddings(pixel_values=pixel_values)
        hidden = self.vision.pre_layrnorm(hidden)
        selected_cls = []
        last_needed = max(self.selected)

        for index, layer in enumerate(self.vision.encoder.layers):
            if self.training and hidden.requires_grad:
                hidden = checkpoint(
                    lambda x, module=layer: self._run_block(module, x),
                    hidden,
                    use_reentrant=False,
                )
            else:
                hidden = self._run_block(layer, hidden)

            if index in self.selected:
                hidden = self.adapters[str(index)](hidden)
                selected_cls.append(hidden[:, 0])
            if index >= last_needed:
                break

        if len(selected_cls) != len(self.selected):
            raise RuntimeError(f"Expected {len(self.selected)} selected CLIP depths, got {len(selected_cls)}")
        return self.fusion(torch.cat(selected_cls, dim=-1))


class LearnablePathologyAnchors(nn.Module):
    def __init__(self, clip_model, tokenizer, n_ctx=PROMPT_CONTEXT_TOKENS):
        super().__init__()
        self.text_model = clip_model.text_model
        self.text_projection = clip_model.text_projection
        self.n_ctx = int(n_ctx)

        prompts = []
        for negative, positive in ANCHOR_TEXT:
            prompts.extend([
                " ".join(["X"] * self.n_ctx) + " a knee MRI showing " + negative,
                " ".join(["X"] * self.n_ctx) + " a knee MRI showing " + positive,
            ])
        tokens = tokenizer(
            prompts,
            padding="max_length",
            truncation=True,
            max_length=int(clip_model.config.text_config.max_position_embeddings),
            return_tensors="pt",
        )
        self.register_buffer("input_ids", tokens["input_ids"], persistent=True)
        self.register_buffer("eot_positions", tokens["input_ids"].argmax(dim=-1), persistent=True)

        with torch.no_grad():
            embedded = self.text_model.embeddings.token_embedding(tokens["input_ids"])
            initial_context = embedded[:, 1:1 + self.n_ctx].clone()
        self.context = nn.Parameter(initial_context)

    def _causal_mask(self, batch, length, dtype, device):
        mask = torch.full(
            (batch, 1, length, length),
            torch.finfo(dtype).min,
            device=device,
            dtype=dtype,
        )
        return torch.triu(mask, diagonal=1)

    def forward(self):
        token_embedding = self.text_model.embeddings.token_embedding(self.input_ids).detach().clone()
        token_embedding[:, 1:1 + self.n_ctx] = self.context
        length = token_embedding.shape[1]
        position_ids = torch.arange(length, device=token_embedding.device).unsqueeze(0)
        hidden = token_embedding + self.text_model.embeddings.position_embedding(position_ids)
        causal_mask = self._causal_mask(hidden.shape[0], length, hidden.dtype, hidden.device)
        encoder_parameters = inspect.signature(self.text_model.encoder.forward).parameters
        if "causal_attention_mask" in encoder_parameters:
            encoder_output = self.text_model.encoder(
                inputs_embeds=hidden,
                attention_mask=None,
                causal_attention_mask=causal_mask,
                output_attentions=False,
                output_hidden_states=False,
                return_dict=True,
            )
        else:
            encoder_output = self.text_model.encoder(
                inputs_embeds=hidden,
                attention_mask=causal_mask,
                is_causal=True,
            )
        encoded = encoder_output.last_hidden_state
        encoded = self.text_model.final_layer_norm(encoded)
        pooled = encoded[torch.arange(encoded.shape[0], device=encoded.device), self.eot_positions]
        projected = self.text_projection(pooled)
        return projected.view(len(LABEL_COLS), 2, -1)


class AdaptiveCLIPMIL12(nn.Module):
    def __init__(self, model_dir, tokenizer):
        super().__init__()
        clip = CLIPModel.from_pretrained(str(model_dir), local_files_only=True)
        for parameter in clip.parameters():
            parameter.requires_grad = False

        self.visual = MultiDepthCLIPVisual(clip)
        self.anchors = LearnablePathologyAnchors(clip, tokenizer)
        width = self.visual.width
        clip_projection_dim = int(clip.config.projection_dim)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(width, len(LABEL_COLS))
        self.visual_alignment = nn.Linear(width, ALIGNMENT_DIM, bias=False)
        self.text_alignment = nn.Linear(clip_projection_dim, ALIGNMENT_DIM, bias=False)

    def encode_slice_chunk(self, chunk):
        mean = CLIP_MEAN.to(chunk.device)
        std = CLIP_STD.to(chunk.device)
        pixels = chunk.repeat(1, 3, 1, 1)
        pixels = (pixels - mean) / std
        return self.visual(pixels)

    def encode_series(self, volume):
        chunk_features = []
        for start in range(0, len(volume), SLICE_CHUNK):
            chunk_features.append(self.encode_slice_chunk(volume[start:start + SLICE_CHUNK]))
        slice_features = torch.cat(chunk_features, dim=0)
        return slice_features.max(dim=0).values

    def encode_study(self, volumes):
        if not 1 <= len(volumes) <= SERIES_PER_STUDY:
            raise ValueError(f"Expected one to {SERIES_PER_STUDY} series, received {len(volumes)}")
        series_features = torch.stack([self.encode_series(volume) for volume in volumes], dim=0)
        return series_features.max(dim=0).values

    def forward(self, volumes):
        study = self.encode_study(volumes)
        return self.classifier(self.dropout(study))

    def forward_training(self, volumes):
        study = self.encode_study(volumes)
        logits = self.classifier(self.dropout(study))

        visual = F.normalize(self.visual_alignment(study), dim=-1)
        anchors = F.normalize(self.text_alignment(self.anchors()), dim=-1)
        similarities = torch.einsum("d,cpd->cp", visual, anchors)
        anchor_logits = (similarities[:, 1] - similarities[:, 0]) / ALIGNMENT_TEMPERATURE
        return logits, anchor_logits

    def parameter_efficient_l2(self):
        penalties = [p.float().pow(2).mean() for p in self.parameters() if p.requires_grad]
        return torch.stack(penalties).mean() if penalties else torch.tensor(0.0, device=next(self.parameters()).device)


## 6. Build labelled and unlabelled study loaders

Gold studies receive stochastic strong views and true labels. For every unlabelled study, the EMA teacher receives deterministic bin-centre slices while the student receives a separately sampled and perturbed view of the same three series.


In [7]:
# Select three readable, plane-diverse series for all 46 labelled development studies.
gold_series_candidates = series_meta[
    series_meta["StudyInstanceUID"].astype(str).isin(set(gold_dev["StudyInstanceUID"].astype(str)))
].copy()
gold_series_ok = []
for row in gold_series_candidates.itertuples(index=False):
    directory = TRAIN_SERIES_ROOT / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID)
    try:
        ok = directory.exists() and any(
            path.is_file() and not path.name.startswith(".")
            for path in directory.iterdir()
        )
    except Exception:
        ok = False
    gold_series_ok.append(ok)
gold_series_candidates = gold_series_candidates.loc[gold_series_ok].copy()
gold_series_candidates["signature_rank"] = gold_series_candidates["signature"].map(signature_rank).fillna(len(signature_rank) + 1)
gold_series_candidates["target_first"] = (gold_series_candidates["signature"] != TARGET_SIGNATURE).astype(int)
gold_series_selection = choose_plane_diverse_series(
    gold_series_candidates.drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
).reset_index(drop=True)

gold_series_counts = gold_series_selection.groupby("StudyInstanceUID").size()
complete_gold_ids = set(gold_series_counts[gold_series_counts == SERIES_PER_STUDY].index.astype(str))
missing_gold_ids = set(gold_dev["StudyInstanceUID"].astype(str)) - complete_gold_ids
if missing_gold_ids:
    raise RuntimeError(f"Development-gold studies without three readable series: {sorted(missing_gold_ids)}")

all_training_series = pd.concat(
    [gold_series_selection, unlabelled_series_selection],
    ignore_index=True,
    sort=False,
).drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])

if set(gold_final["StudyInstanceUID"].astype(str)) & set(all_training_series["StudyInstanceUID"].astype(str)):
    raise RuntimeError("Final-gold holdout leaked into semi-supervised training")

def make_series_map(selected_series):
    return {
        str(study_id): group.sort_values(["target_first", "signature_rank", "SeriesInstanceUID"])
        for study_id, group in selected_series.groupby("StudyInstanceUID")
    }

def perturb_volume(volume):
    x = volume.clone()
    if np.random.random() < 0.5:
        x = torch.flip(x, dims=[-1])
    scale = float(np.random.uniform(0.85, 1.15))
    shift = float(np.random.uniform(-0.08, 0.08))
    gamma = float(np.random.uniform(0.85, 1.20))
    x = torch.clamp(x * scale + shift, 0.0, 1.0).pow(gamma)
    if np.random.random() < 0.5:
        x = torch.clamp(x + 0.02 * torch.randn_like(x), 0.0, 1.0)
    if len(x) > 1 and np.random.random() < 0.20:
        x[np.random.randint(0, len(x))] = 0.0
    return x

class LabelledStudyDataset(Dataset):
    def __init__(self, frame, selected_series, series_root):
        self.frame = frame.reset_index(drop=True)
        self.series_map = make_series_map(selected_series)
        self.series_root = series_root

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        study_id = str(row.StudyInstanceUID)
        selected = self.series_map[study_id]
        volumes = [
            perturb_volume(load_series_volume(
                self.series_root, study_id, series_id, stochastic=True
            ))
            for series_id in selected["SeriesInstanceUID"].astype(str)
        ]
        target = torch.tensor(row[LABEL_COLS].astype(float).to_numpy(), dtype=torch.float32)
        return {"volumes": volumes, "target": target, "study_id": study_id}

class UnlabelledStudyDataset(Dataset):
    def __init__(self, frame, selected_series, series_root):
        self.frame = frame.reset_index(drop=True)
        self.series_map = make_series_map(selected_series)
        self.series_root = series_root

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        study_id = str(self.frame.iloc[index].StudyInstanceUID)
        selected = self.series_map[study_id]
        teacher_volumes, student_volumes = [], []
        for series_id in selected["SeriesInstanceUID"].astype(str):
            teacher_volumes.append(load_series_volume(
                self.series_root, study_id, series_id, stochastic=False
            ))
            student_volumes.append(perturb_volume(load_series_volume(
                self.series_root, study_id, series_id, stochastic=True
            )))
        return {
            "teacher_volumes": teacher_volumes,
            "student_volumes": student_volumes,
            "study_id": study_id,
        }

def collate_single(batch):
    return batch[0]

labelled_loader = DataLoader(
    LabelledStudyDataset(gold_dev, gold_series_selection, TRAIN_SERIES_ROOT),
    batch_size=1, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=AMP, collate_fn=collate_single,
)
unlabelled_loader = DataLoader(
    UnlabelledStudyDataset(selected_unlabelled_studies, unlabelled_series_selection, TRAIN_SERIES_ROOT),
    batch_size=1, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=AMP, collate_fn=collate_single,
)

print("Labelled gold studies:", len(gold_dev))
print("Unlabelled MRI studies:", len(selected_unlabelled_studies))
print("Final holdout studies:", len(gold_final))


Labelled gold studies: 46
Unlabelled MRI studies: 1000
Final holdout studies: 12


## 7. Gold warm-up followed by Mean Teacher training

The student first learns from the 46 gold studies. The EMA teacher is then initialized from that warm-up state. During joint training, pathology classification and text-anchor alignment remain gold-only; unlabelled studies contribute only confidence-weighted agreement between weak teacher views and strong student views.


In [8]:
seed_everything(SEED + 500)
clip_tokenizer = AutoTokenizer.from_pretrained(
    str(CLIP_MODEL_DIR), local_files_only=True, use_fast=True
)
student_model = AdaptiveCLIPMIL12(CLIP_MODEL_DIR, clip_tokenizer).to(DEVICE)
trainable_parameters = [p for p in student_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters, lr=IMAGE_LR, weight_decay=IMAGE_WEIGHT_DECAY
)
scaler = torch.cuda.amp.GradScaler(enabled=AMP)

total_parameters = sum(p.numel() for p in student_model.parameters())
trainable_count = sum(p.numel() for p in trainable_parameters)
print(f"Trainable parameters: {trainable_count:,}/{total_parameters:,} ({100 * trainable_count / total_parameters:.3f}%)")

def supervised_objective(model, volumes, target):
    logits, anchor_logits = model.forward_training(volumes)
    classification = F.binary_cross_entropy_with_logits(logits, target)
    alignment = F.binary_cross_entropy_with_logits(anchor_logits, target)
    regularization = model.parameter_efficient_l2()
    total = classification + ALIGNMENT_BETA * alignment + PE_L2_COEFF * regularization
    return total, classification, alignment

history = []

# Stage 1: gold-only warm-up so the EMA teacher begins from a meaningful model.
for epoch in range(1, WARMUP_EPOCHS + 1):
    student_model.train()
    epoch_total, epoch_cls, epoch_align = [], [], []
    for batch in labelled_loader:
        volumes = [volume.to(DEVICE, non_blocking=True) for volume in batch["volumes"]]
        target = batch["target"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            total, classification, alignment = supervised_objective(student_model, volumes, target)
        scaler.scale(total).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_total.append(float(total.detach().cpu()))
        epoch_cls.append(float(classification.detach().cpu()))
        epoch_align.append(float(alignment.detach().cpu()))

    row = {
        "stage": "gold_warmup", "epoch": epoch,
        "total_loss": float(np.mean(epoch_total)),
        "supervised_bce": float(np.mean(epoch_cls)),
        "anchor_bce": float(np.mean(epoch_align)),
        "consistency_bce": np.nan,
        "accepted_unlabelled_fraction": np.nan,
    }
    history.append(row)
    print(row)

# Stage 2: EMA Mean Teacher consistency on 1,000 unlabelled MRIs.
ema_teacher = copy.deepcopy(student_model).to(DEVICE)
ema_teacher.eval()
for parameter in ema_teacher.parameters():
    parameter.requires_grad = False

@torch.no_grad()
def update_ema(teacher, student, decay):
    student_parameters = dict(student.named_parameters())
    for name, teacher_parameter in teacher.named_parameters():
        student_parameter = student_parameters[name]
        if student_parameter.requires_grad:
            teacher_parameter.mul_(decay).add_(student_parameter.detach(), alpha=1.0 - decay)

total_ssl_steps = max(1, SSL_EPOCHS * len(unlabelled_loader))
global_ssl_step = 0
ssl_started_at = time.time()

for epoch in range(1, SSL_EPOCHS + 1):
    student_model.train()
    ema_teacher.eval()
    labelled_iterator = iter(labelled_loader)
    epoch_total, epoch_sup, epoch_consistency, accepted = [], [], [], []

    for step, unlabelled_batch in enumerate(unlabelled_loader, 1):
        try:
            labelled_batch = next(labelled_iterator)
        except StopIteration:
            labelled_iterator = iter(labelled_loader)
            labelled_batch = next(labelled_iterator)

        labelled_volumes = [v.to(DEVICE, non_blocking=True) for v in labelled_batch["volumes"]]
        target = labelled_batch["target"].to(DEVICE, non_blocking=True)
        teacher_volumes = [v.to(DEVICE, non_blocking=True) for v in unlabelled_batch["teacher_volumes"]]
        student_volumes = [v.to(DEVICE, non_blocking=True) for v in unlabelled_batch["student_volumes"]]

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=AMP):
            supervised_loss, _, _ = supervised_objective(student_model, labelled_volumes, target)
        scaler.scale(supervised_loss).backward()

        with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP):
            teacher_probability = torch.sigmoid(ema_teacher(teacher_volumes)).float()

        with torch.cuda.amp.autocast(enabled=AMP):
            student_logits = student_model(student_volumes)
            confidence_mask = (
                (teacher_probability >= POSITIVE_CONFIDENCE)
                | (teacher_probability <= NEGATIVE_CONFIDENCE)
            )
            confidence = 2.0 * torch.abs(teacher_probability - 0.5)
            per_label = F.binary_cross_entropy_with_logits(
                student_logits, teacher_probability.to(student_logits.dtype), reduction="none"
            )
            weights = confidence_mask.to(per_label.dtype) * confidence.to(per_label.dtype)
            if bool(confidence_mask.any()):
                consistency_loss = (per_label * weights).sum() / weights.sum().clamp_min(1e-6)
            else:
                consistency_loss = student_logits.sum() * 0.0

            ramp_fraction = min(1.0, (global_ssl_step + 1) / max(1.0, 0.4 * total_ssl_steps))
            unlabelled_weight = UNLABELED_WEIGHT_MAX * ramp_fraction
            weighted_consistency = unlabelled_weight * consistency_loss

        scaler.scale(weighted_consistency).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        update_ema(ema_teacher, student_model, EMA_DECAY)

        combined = float((supervised_loss.detach() + weighted_consistency.detach()).cpu())
        epoch_total.append(combined)
        epoch_sup.append(float(supervised_loss.detach().cpu()))
        epoch_consistency.append(float(consistency_loss.detach().cpu()))
        accepted.append(float(confidence_mask.float().mean().cpu()))
        global_ssl_step += 1

        if step % 50 == 0 or step == len(unlabelled_loader):
            elapsed = time.time() - ssl_started_at
            remaining_hours = (elapsed / max(1, global_ssl_step)) * (total_ssl_steps - global_ssl_step) / 3600
            print(
                f"ssl epoch {epoch}/{SSL_EPOCHS} | {step}/{len(unlabelled_loader)} "
                f"| total {np.mean(epoch_total[-50:]):.4f} "
                f"| accepted {np.mean(accepted[-50:]):.3f} "
                f"| estimated remaining {remaining_hours:.2f} h"
            )

    row = {
        "stage": "mean_teacher", "epoch": epoch,
        "total_loss": float(np.mean(epoch_total)),
        "supervised_bce": float(np.mean(epoch_sup)),
        "anchor_bce": np.nan,
        "consistency_bce": float(np.mean(epoch_consistency)),
        "accepted_unlabelled_fraction": float(np.mean(accepted)),
    }
    history.append(row)
    print(row)

# Use the EMA teacher for final evaluation and hidden inference.
image_model = ema_teacher
image_model.eval()
best_image_state = {
    key: value.detach().cpu().clone()
    for key, value in image_model.state_dict().items()
}

# Release the student copy before final-holdout and hidden inference.
del student_model, optimizer, scaler, trainable_parameters
gc.collect()
if AMP:
    torch.cuda.empty_cache()

pd.DataFrame(history).to_csv(OUT / "mean_teacher_training_history.csv", index=False)
torch.save(best_image_state, OUT / "adaptive_clip_mean_teacher_1046studies_final.pt")
display(pd.DataFrame(history))

fallback_probabilities = torch.tensor(
    gold_dev[LABEL_COLS].mean().to_numpy(np.float32), dtype=torch.float32
)
bundle = {
    "format_version": 4,
    "architecture": "adaptive_clip_vit_b16_mil12_mean_teacher",
    "state_dict": best_image_state,
    "label_cols": list(LABEL_COLS),
    "fallback_probabilities": fallback_probabilities,
    "target_signature": str(TARGET_SIGNATURE),
    "signature_rank": {str(k): int(v) for k, v in signature_rank.items()},
    "image_size": int(IMAGE_SIZE),
    "max_slices": int(MAX_SLICES),
    "series_per_study": int(SERIES_PER_STUDY),
    "adapter_blocks_one_indexed": [int(x) + 1 for x in ADAPTER_LAYER_INDICES],
    "n_gold_train": int(len(gold_dev)),
    "n_unlabelled_train": int(len(selected_unlabelled_studies)),
    "ema_decay": float(EMA_DECAY),
    "positive_confidence": float(POSITIVE_CONFIDENCE),
    "negative_confidence": float(NEGATIVE_CONFIDENCE),
}
torch.save(bundle, OUT / "rsna_adaptive_clip_mean_teacher_inference_bundle.pt")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: /kaggle/input/models/amritgopinath/clip-vit-model/pytorch/default/1
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 3,371,599/152,599,119 (2.209%)
{'stage': 'gold_warmup', 'epoch': 1, 'total_loss': 0.712622203256773, 'supervised_bce': 0.6484040332877118, 'anchor_bce': 0.6421816575786342, 'consistency_bce': nan, 'accepted_unlabelled_fraction': nan}
{'stage': 'gold_warmup', 'epoch': 2, 'total_loss': 0.6793083295873974, 'supervised_bce': 0.6164577408977177, 'anchor_bce': 0.6285058616296105, 'consistency_bce': nan, 'accepted_unlabelled_fraction': nan}
ssl epoch 1/1 | 50/1000 | total 0.7112 | accepted 0.407 | estimated remaining 0.30 h
ssl epoch 1/1 | 100/1000 | total 0.7832 | accepted 0.410 | estimated remaining 0.28 h
ssl epoch 1/1 | 150/1000 | total 0.8143 | accepted 0.387 | estimated remaining 0.27 h
ssl epoch 1/1 | 200/1000 | total 0.9017 | accepted 0.402 | estimated remaining 0.26 h
ssl epoch 1/1 | 250/1000 | total 0.9465 | accepted 0.390 | estimated remaining 0.24 h
ssl epoch 1/1 | 300/1000 | total 1.0036 | accepted 0.402 | estimated remaining 0.22 h
ssl epoch 1/1 | 350/1000 

,stage,epoch,total_loss,supervised_bce,anchor_bce,consistency_bce,accepted_unlabelled_fraction
0,gold_warmup,1,0.712622,0.648404,0.642182,NaN,NaN
1,gold_warmup,2,0.679308,0.616458,0.628506,NaN,NaN
2,mean_teacher,1,1.019902,0.625149,NaN,0.495557,0.440833


## 8. One-time evaluation on the untouched 12-study gold holdout

The final EMA teacher is evaluated once. These 12 studies are absent from warm-up, Mean Teacher training, confidence tuning, and selection.


In [9]:
def attach_signatures(frame):
    x = frame.copy()
    for col in ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]:
        if col not in x.columns: x[col] = "UNKNOWN"
    x["plane_norm"] = x["Anatomical_Plane"].map(canonical_text).replace({"SAG":"SAGITTAL", "COR":"CORONAL", "AX":"AXIAL"})
    x["fluid_norm"] = x["Fluid_Sensitive"].map(boolish_text)
    x["fat_norm"] = x["Fat_Suppression"].map(boolish_text)
    x["signature"] = x[["plane_norm", "fluid_norm", "fat_norm"]].agg("|".join, axis=1)
    x["signature_rank"] = x["signature"].map(signature_rank).fillna(len(signature_rank) + 1)
    x["target_first"] = (x["signature"] != TARGET_SIGNATURE).astype(int)
    return x

def select_three_series_per_study(series_frame, study_ids, series_root):
    x = attach_signatures(series_frame)
    x = x[x.StudyInstanceUID.astype(str).isin(set(map(str, study_ids)))].copy()
    ok = []
    for r in x.itertuples(index=False):
        d = series_root / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
        ok.append(d.exists() and any(p.is_file() and not p.name.startswith(".") for p in d.iterdir()))
    x = x.loc[ok]
    x = x.sort_values(["StudyInstanceUID", "target_first", "signature_rank", "SeriesInstanceUID"])
    x = x.drop_duplicates(["StudyInstanceUID", "SeriesInstanceUID"])
    return choose_plane_diverse_series(x).reset_index(drop=True)

@torch.inference_mode()
def predict_selected_studies(selected, series_root):
    records = []
    groups = list(selected.groupby("StudyInstanceUID", sort=True))
    for i, (study_id, group) in enumerate(groups, 1):
        if len(group) != SERIES_PER_STUDY:
            raise ValueError(f"Study {study_id} has {len(group)} readable selected series")
        volumes = [
            load_series_volume(series_root, study_id, series_id).to(DEVICE)
            for series_id in group.SeriesInstanceUID.astype(str)
        ]
        with torch.cuda.amp.autocast(enabled=AMP):
            prob = torch.sigmoid(image_model(volumes)).float().cpu().numpy()
        rec = {
            "StudyInstanceUID": str(study_id),
            "SeriesInstanceUIDs": "|".join(group.SeriesInstanceUID.astype(str)),
            "n_target_signature_series": int((group.signature == TARGET_SIGNATURE).sum()),
        }
        rec.update({f"pred__{label}": float(prob[j]) for j, label in enumerate(LABEL_COLS)})
        records.append(rec)
        if i % 25 == 0 or i == len(groups):
            print(f"Predicted {i}/{len(groups)} studies")
    return pd.DataFrame(records)

final_selected = select_three_series_per_study(
    train_series, gold_final.StudyInstanceUID, TRAIN_SERIES_ROOT
)
final_counts = final_selected.groupby("StudyInstanceUID").size()
complete_final_ids = set(final_counts[final_counts == SERIES_PER_STUDY].index)
missing_final = set(gold_final.StudyInstanceUID) - complete_final_ids
if missing_final:
    raise RuntimeError(f"Final holdout studies without three readable series: {sorted(missing_final)}")

final_pred = predict_selected_studies(final_selected, TRAIN_SERIES_ROOT)
final_eval = gold_final[["StudyInstanceUID"] + LABEL_COLS].merge(final_pred, on="StudyInstanceUID", how="inner")
y_final = final_eval[LABEL_COLS].to_numpy(np.float32)
p_final = final_eval[[f"pred__{x}" for x in LABEL_COLS]].to_numpy(np.float32)
final_metrics = multilabel_metrics(y_final, p_final)
final_metrics.to_csv(OUT / "final_gold_12_metrics.csv", index=False)
final_eval.to_csv(OUT / "final_gold_12_predictions.csv", index=False)
print("Final holdout studies evaluated with exactly three series:", len(final_pred))
display(final_metrics)

Predicted 12/12 studies
Final holdout studies evaluated with exactly three series: 12


,pathology,n,positives,roc_auc,pr_auc,f1_at_0.5
0,ACL,12,5,0.685714,0.586667,0.285714
1,MCL,12,2,0.600000,0.361111,0.000000
2,Medial Meniscus,12,6,0.638889,0.697391,0.545455
3,Lateral Meniscus,12,5,0.571429,0.663889,0.333333
4,Medial OA,12,3,0.777778,0.680556,0.000000
5,Lateral OA,12,2,0.900000,0.583333,0.000000
6,PF OA,12,5,0.828571,0.890909,0.000000
7,Effusion,12,7,0.457143,0.693682,0.533333
8,Synovitis,12,6,0.722222,0.840909,0.727273
9,Baker's,12,2,0.700000,0.625000,0.000000


## 9. Hidden-test inference and exact `submission.csv`

The hidden study IDs and final column order come only from `sample_submission.csv`. Individual series/study failures are recorded without terminating the rerun. Per-pathology prevalence from the 46 gold development studies is used only as the failure fallback.


In [10]:
def candidate_test_roots():
    candidates = [
        ROOT / "test_series",
        ROOT / "test",
        ROOT / "test_images",
        TRAIN_SERIES_ROOT,
    ]
    return [path for path in candidates if path.exists() and path.is_dir()]

def choose_test_series_root(study_ids):
    ids = [str(x) for x in study_ids]
    candidates = candidate_test_roots()
    # Prefer the directory that actually contains one of the submitted study IDs.
    for directory in candidates:
        if any((directory / sid).is_dir() for sid in ids[: min(25, len(ids))]):
            return directory
    # Public placeholder IDs may have no public image directories. Keep a deterministic
    # existing root so that a valid fallback submission can still be produced.
    return candidates[0] if candidates else None

def load_test_series_metadata(test_root, study_ids):
    ids = set(map(str, study_ids))
    csv_candidates = [
        ROOT / "test_series.csv",
        ROOT / "series.csv",
        ROOT / "train_series.csv",
    ]
    for csv_path in csv_candidates:
        if not csv_path.exists():
            continue
        frame = pd.read_csv(
            csv_path,
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        ).drop_duplicates()
        overlap = frame["StudyInstanceUID"].astype(str).isin(ids).any()
        if overlap:
            print("Using test-series metadata:", csv_path)
            return frame

    # Metadata-free fallback: enumerate study/series folders from sample IDs.
    rows = []
    if test_root is not None:
        for study_id in sorted(ids):
            study_dir = test_root / study_id
            if not study_dir.exists():
                continue
            try:
                series_dirs = sorted(path for path in study_dir.iterdir() if path.is_dir())
            except Exception:
                series_dirs = []
            for series_dir in series_dirs:
                rows.append({
                    "StudyInstanceUID": study_id,
                    "SeriesInstanceUID": series_dir.name,
                    "Anatomical_Plane": "UNKNOWN",
                    "Fluid_Sensitive": "UNKNOWN",
                    "Fat_Suppression": "UNKNOWN",
                })
    print("Enumerated test-series metadata rows:", len(rows))
    return pd.DataFrame(rows, columns=[
        "StudyInstanceUID", "SeriesInstanceUID", "Anatomical_Plane",
        "Fluid_Sensitive", "Fat_Suppression",
    ])

sample_submission = pd.read_csv(
    ROOT / "sample_submission.csv",
    dtype={"StudyInstanceUID": str},
)
if "StudyInstanceUID" not in sample_submission.columns:
    raise ValueError("sample_submission.csv has no StudyInstanceUID column")

submission_labels = [column for column in sample_submission.columns if column != "StudyInstanceUID"]
missing_outputs = [column for column in submission_labels if column not in LABEL_COLS]
if missing_outputs:
    raise ValueError(f"Model has no output for submission columns: {missing_outputs}")

test_ids = sample_submission["StudyInstanceUID"].astype(str).tolist()
TEST_SERIES_ROOT = choose_test_series_root(test_ids)
test_series = load_test_series_metadata(TEST_SERIES_ROOT, test_ids)

if TEST_SERIES_ROOT is None:
    print("SEVERE WARNING: no MRI series root was found; all rows will use fallback probabilities")
else:
    print("Test MRI root:", TEST_SERIES_ROOT)

test_selected = select_three_series_per_study(
    test_series,
    test_ids,
    TEST_SERIES_ROOT,
) if TEST_SERIES_ROOT is not None else pd.DataFrame()

fallback = gold_dev[LABEL_COLS].mean().to_numpy(np.float32)
pred_map = {study_id: fallback.copy() for study_id in test_ids}
failures = []

image_model.eval()
for index, study_id in enumerate(test_ids, 1):
    if test_selected.empty:
        group = pd.DataFrame()
    else:
        group = test_selected[
            test_selected["StudyInstanceUID"].astype(str) == study_id
        ]

    volumes = []
    errors = []
    if not group.empty:
        for series_id in group["SeriesInstanceUID"].astype(str):
            try:
                volume = load_series_volume(
                    TEST_SERIES_ROOT,
                    study_id,
                    series_id,
                    max_slices=MAX_SLICES,
                ).to(DEVICE)
                volumes.append(volume)
            except Exception as exc:
                errors.append(f"{series_id}: {type(exc).__name__}: {exc}")

    if volumes:
        try:
            with torch.inference_mode(), torch.cuda.amp.autocast(enabled=AMP):
                probabilities = torch.sigmoid(image_model(volumes)).float().cpu().numpy()
            probabilities = np.where(np.isfinite(probabilities), probabilities, fallback)
            pred_map[study_id] = np.clip(probabilities, 0.0, 1.0)
        except Exception as exc:
            errors.append(f"model: {type(exc).__name__}: {exc}")
    else:
        errors.append("No decodable selected MRI series; used per-label fallback")

    if errors:
        failures.append({
            "StudyInstanceUID": study_id,
            "n_selected_series": int(len(group)),
            "n_decoded_series": int(len(volumes)),
            "error": " || ".join(errors),
        })
    if index % 25 == 0 or index == len(test_ids):
        print(f"Test inference {index}/{len(test_ids)}")

submission = sample_submission[["StudyInstanceUID"]].copy()
prediction_matrix = np.stack([pred_map[study_id] for study_id in test_ids])
for label in submission_labels:
    submission[label] = prediction_matrix[:, LABEL_COLS.index(label)]

values = submission[submission_labels].to_numpy(dtype=np.float64)
values = np.clip(np.nan_to_num(values, nan=0.5, posinf=1.0, neginf=0.0), 0.0, 1.0)
submission.loc[:, submission_labels] = values
submission = submission.reindex(columns=sample_submission.columns)

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
FAILURE_PATH = OUT / "test_inference_failures.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
pd.DataFrame(
    failures,
    columns=["StudyInstanceUID", "n_selected_series", "n_decoded_series", "error"],
).to_csv(FAILURE_PATH, index=False)

print("Saved:", SUBMISSION_PATH)
print("Submission shape:", submission.shape)
print("Studies with any recorded issue:", len(failures))
print("Studies using complete fallback:", sum(row["n_decoded_series"] == 0 for row in failures))
display(submission.head())


Using test-series metadata: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv
Test MRI root: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
Test inference 3/3
Saved: /kaggle/working/submission.csv
Submission shape: (3, 13)
Studies with any recorded issue: 0
Studies using complete fallback: 0


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.399658,0.175171,0.584473,0.450195,0.308594,0.375000,0.397705,0.484619,0.538574,0.166382,0.271729,0.270996
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.385254,0.221069,0.410400,0.217285,0.227295,0.173828,0.293945,0.419922,0.393799,0.178345,0.442871,0.225464
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.336426,0.198486,0.495117,0.416504,0.204834,0.225464,0.363281,0.684570,0.447510,0.083435,0.166626,0.377686


## Submission checklist

- Attach the RSNA competition data.
- Attach the full offline `openai/clip-vit-base-patch16` model at the configured path.
- KBioXLM and AlexNet are **not used**.
- Enable GPU and disable Internet.
- Use **Save Version → Save & Run All**.
- Submit `/kaggle/working/submission.csv`.

The public placeholder rows may use fallback probabilities if Kaggle exposes no public test MRI folders. During scoring, Kaggle substitutes the hidden dataset and reruns this notebook.
